# Урок 6. Многоклассовая классификация.

Посмотрим на примере алгоритма логистической регрессии и метода опорных векторов, как работать с различными методами многоклассовой классификации.

### 1.
Вспомните датасет Wine. Загрузите его, разделите на тренировочную и тестовую выборки (random_state=17), используя только [9, 11, 12] признаки.

In [1]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

In [3]:
### YOUR CODE HERE ###
wine_dataset = load_wine()
x_train, x_test, y_train, y_test = train_test_split(wine_dataset.data[:, [9, 11, 12]],
                                                    wine_dataset['target'],
                                                    random_state=17)




**Задайте тип кросс-валидации с помощью StratifiedKFold: 5-кратная, random_state=17.**

In [4]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [8]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=17)



### 2.
Обучите логистическую регрессию (LogisticRegression) с параметром C по умолчанию и random_state=17. Укажите гиперпараметр multi_class='ovr' - по умолчанию многие классификаторы используют именно его. С помощью cross_val_score сделайте кросс-валидацию (используйте объект skf) и выведите среднюю долю правильных ответов на ней (используйте функцию mean). Отдельно выведите долю правильных ответов на тестовой выборке.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [14]:
### YOUR CODE HERE ###
lr = LogisticRegression(C=1.0, random_state=17, multi_class='ovr')
model = lr.fit(x_train, y_train)

cv_scores = cross_val_score(model, x_train, y_train, cv=skf, scoring='accuracy')

# Средняя точность по кросс-валидации
print("Средняя точность (CV):", cv_scores.mean())

y_pred_logreg = model.predict(x_test)

# Точность на тестовой выборке
test_accuracy = accuracy_score(y_test, y_pred_logreg)
print("Точность на тестовой выборке:", test_accuracy)

Средняя точность (CV): 0.9096866096866097
Точность на тестовой выборке: 0.9111111111111111


C:\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be rem

### 3.
Обучите метод опорных векторов (SVC) с random_state=17 и остальными параметрами по умолчанию. Этот метод при мультиклассовой классификации также использует метод "ovr". Сделайте кросс-валидацию (используйте skf) и, как и в предыдущем пункте, выведите среднюю долю правильных ответов на ней. Отдельно выведите долю правильных ответов на тестовой выборке.

In [12]:
from sklearn.svm import SVC

In [15]:
### YOUR CODE HERE ###
svc = SVC(random_state=17)

# Кросс-валидация на тренировочной выборке
svc_cv_scores = cross_val_score(svc, x_train, y_train, cv=skf, scoring='accuracy')

# Выводим среднюю точность
print("Средняя точность (CV) для SVC:", svc_cv_scores.mean())

svc.fit(x_train, y_train)
y_pred_svc = svc.predict(x_test)
svc_accuracy = accuracy_score(y_test, y_pred_svc)
print("Точность на тестовой выборке (SVC):", svc_accuracy)

Средняя точность (CV) для SVC: 0.6923076923076923
Точность на тестовой выборке (SVC): 0.6222222222222222


Как видно из полученной метрики, на тестовой выборке метод с гиперпараметрами по умолчанию работает явно намного хуже логистической регрессии. В целом, SVM достаточно плохо масштабируется на размер обучающего набора данных (как видно, даже с тремя признаками он работает не очень хорошо), но благодаря возможности выбора различных ядер (функций близости, которые помогают разделять данные) и другим гиперпараметрам SVM можно достаточно точно настроить под определенный вид данных. Подробнее на этом останавливаться в контексте данного урока не будем.

### 4.
Для предсказаний обеих моделей постройте матрицу ошибок (confusion matrix) и напишите, какие классы каждая из моделей путает больше всего между собой.

In [17]:
from sklearn.metrics import classification_report, confusion_matrix

In [19]:
### YOUR CODE HERE ###
cm_logreg = confusion_matrix(y_test, y_pred_logreg)

# Classification report
print("📊 Classification Report: Logistic Regression")
print(classification_report(y_test, y_pred_logreg))


cm_svc = confusion_matrix(y_test, y_pred_svc)
# Classification report
print("📊 Classification Report: SVC")
print(classification_report(y_test, y_pred_svc))


📊 Classification Report: Logistic Regression
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         9
           1       0.83      1.00      0.90        19
           2       1.00      0.76      0.87        17

    accuracy                           0.91        45
   macro avg       0.94      0.92      0.92        45
weighted avg       0.93      0.91      0.91        45

📊 Classification Report: SVC
              precision    recall  f1-score   support

           0       0.82      1.00      0.90         9
           1       0.56      1.00      0.72        19
           2       0.00      0.00      0.00        17

    accuracy                           0.62        45
   macro avg       0.46      0.67      0.54        45
weighted avg       0.40      0.62      0.48        45



C:\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### 5.
Для каждой модели выведите classification report.

In [ ]:
### YOUR CODE HERE ###





